## Notebook Name: Landbird Habitat Transformed
**Medallion Layer: Silver**  

**Purpose:** Transform the Raw Data of the Habitat of the Landbird Bird  into a transformed silver layer table, that ensures that the Data Types of the Columns are correct

**Author:** Matthew Kristanto  

**Date Created:** 5/03/26  

**Last Modified:** 5/03/26  

**Source Table:** [silver_landbird_habitat_transformed](https://adb-7405618007730451.11.azuredatabricks.net/editor/notebooks/152820778023960?o=7405618007730451)


**Notes:**
- Check that the Park Code and Site Code is capitalized
- Checks that each Attribute belongs to the correct data type
- Convert Habitat_Num from String to Integer Type
- Does handling of NULL or Missing Values for each of the attributes
- NULL Values in each attributes per Record would be replaced by "None"
- Checks for any Records that have been duplicated  
- Convert "Nov" from Canopy Cover Percentage into "10", as it is known to be "10".
- Split Canopy_cover_pct into 2 Columns for Start Percentage and End Percentage
- Tree Size Class is also split into 2 Columns for Start and End


In [0]:
### Retrieve the Silver Cleaned Table

df_silver = spark.read.table("silver.silver_landbird_habitat_cleaned")

display(df_silver.limit(5))

In [0]:
from pyspark.sql.functions import col, initcap
from pyspark.sql.types import StringType

### Go through each column and make first letter of each word capitalized

for column in df_silver.columns:
    df_silver = df_silver.withColumnRenamed(column, column.title())


In [0]:
from pyspark.sql.functions import col, upper, split

### Check that the Park Code and the Site Code is Capitalized

def get_not_caps(columnName, dataFrame):
    df_not_caps = dataFrame.filter(
        col(columnName) != upper(col(columnName))
    )

    display(df_not_caps)

In [0]:
get_not_caps("Park_code", df_silver)
get_not_caps("Site_code", df_silver)


In [0]:
### Convert and make the Park Code Capitalized

df_silver = df_silver.withColumn("Park_Code", upper(col("Park_Code")))

get_not_caps("Park_Code", df_silver)

In [0]:
### Convert and make the Site Code Capitalized

df_silver = df_silver.withColumn("Site_Code", upper(col("Site_Code")))

get_not_caps("Site_Code", df_silver)

In [0]:
### Check for the NULL Values in Canopy Cover Percentage
df_canopy_cover_percentage_null_check = df_silver.filter(col("Canopy_Cover_Pct").isNull())

display(df_canopy_cover_percentage_null_check)

In [0]:
### Convert the NULL Values
df_silver = df_silver.fillna({
    "Canopy_Cover_Pct": "0-0",
    "Canopy_Cover_Desc": "None",
    "Tree_Size_Class_Cm": "0-0",
    "Tree_Size_Class_Desc": "None"
})

display(df_silver)

In [0]:
### Convert the Less Than Symbols for the Canopy Cover Percentage
from pyspark.sql.functions import regexp_replace

df_silver = df_silver.withColumn("Canopy_Cover_Pct", regexp_replace(col("Canopy_Cover_Pct"), "<", "0-"))

display(df_silver)

In [0]:
from pyspark.sql.types import IntegerType

### Split the Canopy Cover Percentages into two columns named Canopy Cover Low and Canopy Cover High

df_silver = df_silver.withColumn("Canopy_Cover_Low", split(col("canopy_cover_pct"), "-")[0].cast(StringType()))

df_silver = df_silver.withColumn("Canopy_Cover_High", split(col("canopy_cover_pct"), "-")[1].cast(StringType()))



In [0]:
### Drop the Old Canopy Cover
df_silver = df_silver.drop("canopy_cover_pct")

In [0]:
### Convert the Data Error of "Nov" into "10" for the Canopy Cover Percentage
from pyspark.sql.functions import when, size, regexp_replace, col

df_silver = df_silver.withColumn("Canopy_Cover_Low", regexp_replace(col("Canopy_Cover_Low"), "Nov", "10"))

display(df_silver)

In [0]:
### Remove the CM from the end of the values
from pyspark.sql.functions import regexp_replace

df_silver = df_silver.withColumn(
    "Tree_Size_Class_Cm",
    regexp_replace("Tree_Size_Class_Cm", " cm", "")
)

In [0]:
display(df_silver)

In [0]:
### Convert the Less Than Symbols for the Tree Size Class Cm
from pyspark.sql.functions import regexp_replace

df_silver = df_silver.withColumn("Tree_Size_Class_Cm", regexp_replace(col("Tree_Size_Class_Cm"), "<", "0-"))

In [0]:
### Convert the Greater Than Symbols for the Tree Size Class Cm
from pyspark.sql.functions import regexp_replace, concat, lit

df_silver = df_silver.withColumn(
    "Tree_Size_Class_Cm",
    when(
        col("Tree_Size_Class_Cm").startswith(">"),
        concat(col("Tree_Size_Class_Cm").substr(2, 10), lit("-NULL"))
    ).otherwise(col("Tree_Size_Class_Cm"))
)

In [0]:
### Split the Tree Size Class Cm into Two Columns, which is Min and Max
df_silver = df_silver.withColumn("Tree_Size_Class_Min", split(col("Tree_Size_Class_Cm"),"-")[0])

df_silver = df_silver.withColumn("Tree_Size_Class_Max", split(col("Tree_Size_Class_Cm"),"-")[1])

### Remove Old Attribute
df_silver = df_silver.drop("Tree_Size_Class_Cm")

In [0]:
display(df_silver)

In [0]:
### Check that if the Max Value is NULL (String Type) then convert to actual NULL Type

df_silver = df_silver.withColumn(
    "Tree_Size_Class_Max",
    when(col("Tree_Size_Class_Max") == "NULL", None)
    .otherwise(col("Tree_Size_Class_Max"))
)

display(df_silver)

In [0]:
### Check the Data Types of the Attribute
df_silver.printSchema()

In [0]:
### Convert Habitat Number from String to Number
from pyspark.sql.types import BooleanType 
from pyspark.sql.functions import to_date

df_silver = df_silver.withColumn("Habitat_Num", col("Habitat_Num").cast(IntegerType()))

df_silver = df_silver.withColumn("Survey_Date", to_date(col("Survey_Date"), "d/MM/yyyy"))

### Convert Is Forested from String to Boolean
df_silver = df_silver.withColumn("Is_Forested", col("Is_Forested").cast(BooleanType()))

### Convert both Canopy Covers to Integer
df_silver = df_silver.withColumn("Canopy_Cover_Low", col("Canopy_Cover_Low").cast(IntegerType()))

df_silver = df_silver.withColumn("Canopy_Cover_High", col("Canopy_Cover_High").cast(IntegerType()))

### Convert Tree Size Class Min and Max to Integer
df_silver = df_silver.withColumn("Tree_Size_Class_Min", col("Tree_Size_Class_Min").cast(IntegerType()))

df_silver = df_silver.withColumn("Tree_Size_Class_Max", col("Tree_Size_Class_Max").cast(IntegerType()))

df_silver.printSchema()

In [0]:
### Create Silver Schema if not exist
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
display(df_silver)

In [0]:
### Save the Silver Table to the Catalog
(df_silver.write
    .format("delta")  
    .mode("overwrite")  
    .saveAsTable("silver.silver_landbird_habitat_transformed"))

In [0]:
%sql
SELECT * FROM silver.silver_landbird_habitat_transformed